In [12]:
json_folder = "../database/nigeria-clinical-guidelines-dataset/processed_json"
output_path = "../database/scenarios.json"

# Verify
print("Exists:", os.path.exists(json_folder))
print("Files found:", len([f for f in os.listdir(json_folder) if f.endswith(".json")]))

Exists: True
Files found: 270


In [11]:
import json
import os

json_folder = "../database/nigeria-clinical-guidelines-dataset/processed_json"
output_path = "../database/scenarios.json"

def infer_care_level(condition_name, definitive_treatment, complications):
    name = condition_name.lower()
    defn = " ".join(definitive_treatment or []).lower()
    comp = " ".join(complications or []).lower()
    
    tertiary_keywords = [
        "cancer", "tumour", "tumor", "carcinoma", "lymphoma", "leukaemia",
        "leukemia", "surgery", "surgical", "specialist", "icu", "intensive",
        "transplant", "dialysis", "neurosurgery", "cardiac", "teaching hospital",
        "expert", "parenteral", "intravenous", "eclampsia", "stroke", "failure",
        "renal failure", "liver failure", "heart failure", "septic shock",
        "meningitis", "encephalitis", "psychosis", "schizophrenia", "epilepsy",
        "sickle cell", "haemophilia", "glaucoma", "retinal", "hiv", "aids",
        "tuberculosis", "obstetric", "ectopic", "placenta", "haemorrhage"
    ]
    secondary_keywords = [
        "hospital", "admission", "admitted", "inpatient", "general hospital",
        "moderate", "pneumonia", "fracture", "appendix", "hernia", "abscess",
        "malaria", "typhoid", "anaemia", "diabetes", "asthma", "ulcer",
        "hypertension", "wound", "burn", "cellulitis", "pelvic"
    ]
    
    text = name + " " + defn + " " + comp
    
    if any(k in text for k in tertiary_keywords):
        return "tertiary"
    elif any(k in text for k in secondary_keywords):
        return "secondary"
    else:
        return "primary"

def infer_urgency(condition_name, complications):
    name = condition_name.lower()
    comp = " ".join(complications or []).lower()
    text = name + " " + comp
    
    emergency_keywords = [
        "shock", "haemorrhage", "hemorrhage", "eclampsia", "stroke",
        "infarction", "obstruction", "rupture", "ruptured", "acute abdomen",
        "epiglottitis", "anaphylaxis", "poisoning", "overdose", "seizure",
        "meningitis", "encephalitis", "sepsis", "ectopic", "cardiac arrest",
        "respiratory failure", "coma", "unconscious", "trauma"
    ]
    urgent_keywords = [
        "fever", "infection", "pneumonia", "malaria", "typhoid", "appendix",
        "fracture", "burn", "abscess", "cellulitis", "kidney injury",
        "diabetic", "hypertensive", "asthma", "bronchitis", "diarrhoea",
        "dehydration", "anaemia", "jaundice", "hepatitis"
    ]
    
    if any(k in text for k in emergency_keywords):
        return "emergency"
    elif any(k in text for k in urgent_keywords):
        return "urgent"
    else:
        return "routine"

def extract_keywords(condition_name, clinical_features, complications):
    keywords = [condition_name.lower()]
    for feature_group in (clinical_features or []):
        for feature in feature_group.get("features", []):
            words = feature.lower().split()
            keywords.extend([w for w in words if len(w) > 4])
    for comp in (complications or []):
        keywords.extend(comp.lower().split())
    keywords = list(set([k.strip("(),.:;") for k in keywords if len(k) > 3]))
    return keywords[:20]

def build_description(condition_name, introduction, clinical_features):
    desc = introduction or ""
    features = []
    for group in (clinical_features or []):
        features.extend(group.get("features", [])[:3])
    if features:
        feature_text = "; ".join(features[:4])
        desc = f"{desc}. Presents with: {feature_text}" if desc else feature_text
    return desc[:500]

def build_intervention(treatment):
    if not treatment:
        return "Assess and stabilise patient. Refer appropriately."
    parts = []
    non_drug = treatment.get("non_drug", [])
    if non_drug:
        parts.append(non_drug[0])
    drug = treatment.get("drug", [])
    if drug:
        parts.extend(drug[:2])
    definitive = treatment.get("definitive_treatment") or []
    if definitive:
        parts.append(definitive[0])
    return ". ".join(parts)[:600]

# Convert all conditions
scenarios = []
files = [f for f in os.listdir(json_folder) if f.endswith(".json")]
print(f"Found {len(files)} condition files")

for i, filename in enumerate(files):
    filepath = os.path.join(json_folder, filename)
    try:
        with open(filepath, encoding="utf-8") as f:
            data = json.load(f)
        
        condition_name = data.get("condition_name", "")
        clinical_features = data.get("clinical_features", [])
        complications = data.get("complications", [])
        treatment = data.get("treatment", {})
        definitive = data.get("definitive_treatment") or []
        introduction = data.get("introduction", "")
        
        scenario = {
            "id": str(i + 1).zfill(3),
            "condition": condition_name,
            "description": build_description(condition_name, introduction, clinical_features),
            "keywords": extract_keywords(condition_name, clinical_features, complications),
            "care_level": infer_care_level(condition_name, definitive, complications),
            "urgency": infer_urgency(condition_name, complications),
            "intervention": build_intervention(treatment),
            "guideline_source": "NSTG 2022"
        }
        scenarios.append(scenario)
    
    except Exception as e:
        print(f"Error processing {filename}: {e}")

print(f"Successfully converted {len(scenarios)} scenarios")

# Save
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(scenarios, f, indent=2, ensure_ascii=False)
print(f"Saved to {output_path}")

# Sanity check
care_levels = {}
urgencies = {}
for s in scenarios:
    care_levels[s['care_level']] = care_levels.get(s['care_level'], 0) + 1
    urgencies[s['urgency']] = urgencies.get(s['urgency'], 0) + 1

print("\nCare level distribution:", care_levels)
print("Urgency distribution:", urgencies)

# Preview one scenario
print("\nSample scenario:")
print(json.dumps(scenarios[0], indent=2))

Found 270 condition files
Successfully converted 270 scenarios
Saved to ../database/scenarios.json

Care level distribution: {'tertiary': 115, 'secondary': 34, 'primary': 121}
Urgency distribution: {'emergency': 71, 'routine': 137, 'urgent': 62}

Sample scenario:
{
  "id": "001",
  "condition": "Abortion",
  "description": "One of the leading causes of morbidity and mortality among women in Nigeria. Defined as termination of pregnancy prior to age of viability (28 weeks of gestation, however, WHO regards age of viability as 20 weeks' gestation or 500-gm birth-weight). Early abortion occurs before 12 weeks of gestation while late abortion occurs after 12 weeks of gestation. Abortion could either be spontaneous abortion (miscarriage) or induced abortion.. Presents with: Minimal vaginal bleeding with or without lower ",
  "keywords": [
    "reduce",
    "heavy",
    "tissue",
    "recurrent",
    "infertility",
    "endometritis",
    "shock",
    "associated",
    "protruding",
    "regr

In [13]:
# Stopwords to filter out from keywords
stopwords = {
    "with", "and", "the", "for", "from", "that", "this", "have", "been",
    "will", "when", "which", "also", "more", "than", "into", "other",
    "after", "before", "their", "there", "where", "about", "should",
    "given", "noted", "often", "associated", "treatment", "patient",
    "reduce", "occur", "cause", "using", "used", "cases", "refer",
    "referred", "evaluated", "following", "history", "present", "presents",
    "signs", "symptoms", "features", "general", "common", "including",
    "usually", "especially", "known", "based", "related", "level"
}

def extract_keywords_clean(condition_name, clinical_features, complications):
    keywords = [condition_name.lower()]
    
    for feature_group in (clinical_features or []):
        for feature in feature_group.get("features", []):
            words = feature.lower().split()
            keywords.extend([w.strip("(),.:;") for w in words if len(w) > 4 and w.strip("(),.:;") not in stopwords])
    
    for comp in (complications or []):
        words = comp.lower().split()
        keywords.extend([w.strip("(),.:;") for w in words if len(w) > 3 and w.strip("(),.:;") not in stopwords])
    
    keywords = list(set(keywords))
    return keywords[:20]

# Re-run with clean keywords and resave
for i, scenario in enumerate(scenarios):
    filename = files[i]
    filepath = os.path.join(json_folder, filename)
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)
    scenario["keywords"] = extract_keywords_clean(
        data.get("condition_name", ""),
        data.get("clinical_features", []),
        data.get("complications", [])
    )

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(scenarios, f, indent=2, ensure_ascii=False)

print("Keywords cleaned and resaved")
print("Sample keywords:", scenarios[0]["keywords"])

Keywords cleaned and resaved
Sample keywords: ['heavy', 'tissue', 'recurrent', 'infertility', 'endometritis', 'shock', 'protruding', 'regression', 'induced', 'parametritis', 'pain', 'pelvic', 'foetus', 'dyspareunia', 'drainage', 'lower', 'appropriate', 'bleeding', 'causative', 'secondary']
